Step 1: Setup Environment

In [12]:
# Init Minio
!sh /home/LocalLakeHouse/Project/Scripts/1.init_minio.sh "data"


Data subdir: data
Source path: /home/LocalLakeHouse/Project/data
Platform: arm64

mc already exists

Check if the bucket already exists...
]11;?\[2026-04-15 08:24:36 UTC]     0B Bronze/
[2026-04-15 08:24:36 UTC]     0B Gold/
[2026-04-15 08:24:36 UTC]     0B Raw/
[2026-04-15 08:24:36 UTC]     0B Silver/
Bucket already exists

Uploading data from '/home/LocalLakeHouse/Project/data' to 'minio/data-lakehouse/Raw'

...lation.csv: 120.35 MiB / 120.35 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 198.98 MiB/s 0s

In [13]:
# Install necessary packages
import sys
!{sys.executable} -m pip install pyspark
!{sys.executable} -m pip install s3fs
!{sys.executable} -m pip install minio
!{sys.executable} -m pip install pyhive
!{sys.executable} -m pip install trino

In [5]:
# Install dotenv to load environment variables
!{sys.executable} -m pip install python-dotenv

In [6]:
# Load environment variables
import os
from dotenv import load_dotenv
load_dotenv('minio.env')

# Access the environment variables
minio_access_key = os.getenv('MINIO_ACCESS_KEY')
minio_secret_key = os.getenv('MINIO_SECRET_KEY')
minio_endpoint = os.getenv('MINIO_ENDPOINT', "http://minio:9000")
minio_bucket_name = os.getenv('MINIO_BUCKET_NAME', "data-lakehouse")

In [7]:
print("Minio Access Key:", minio_access_key)
print("Minio Secret Key:", minio_secret_key)
print("Minio Endpoint:", minio_endpoint)
print("Minio Bucket Name:", minio_bucket_name)

Minio Access Key: minio_ak
Minio Secret Key: minio_sk
Minio Endpoint: http://minio:9000
Minio Bucket Name: data-lakehouse


In [8]:
# Import Spark libraries
import pyspark
import pyspark.sql.functions as sqlF
from pyspark.sql import SparkSession

In [9]:
# Import SLAlchemy and Pandas libraries
from sqlalchemy.sql import text
from sqlalchemy import create_engine
import pandas as pd

In [10]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("OlistDataIngestion") \
    .config("spark.driver.host", "localhost") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.3") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("hive.metastore.uris", "thrift://metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b0302c7c-b90d-4d3a-9f5f-fe4df9c7e3d1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.3 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 257ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aw

In [11]:
# Check spark configuration
spark.sparkContext.getConf().getAll()

[('spark.eventLog.enabled', 'true'),
 ('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false'),
 ('spark.repl.local.jars',
  'file:///root/.ivy2/jars/io.delta_delta-spark_2.12-3.1.0.jar,file:///root/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.3.jar,file:///root/.ivy2/jars/io.delta_delta-storage-3.1.0.jar,file:///root/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar,file:///root/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.11.1026.jar,file:///root/.ivy2/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar'),
 ('spark.driver.port', '39013'),
 ('spark.app.startTime', '1776241174693'),
 ('spark.hadoop.fs.s3a.aws.credentials.provider',
  'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'),
 ('spark.hadoop.fs.s3a.imp', 'org.apache.hadoop.fs.s3a.S3AFileSystem'),
 ('spark.sql.parquet.int96RebaseModeInWrite', 'CORRECTED'),
 ('spark.serializer', 'org.apache.spark.serializer.KryoSerializer'),
 ('spark.hadoop.fs.s3a.path.style.access', 'True'),
 ('spark.files',
  'file:///root/.ivy2/jars/io

Step 2: Build the data frames

In [19]:
df_test = spark.read.csv("s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv", header=True)
df_test.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  